# 1. What is CRUD?

**CRUD** represents the four basic operations for managing data in a database:

| Operation | SQL Command | Description |
|-----------|-------------|-------------|
| **C**reate | `INSERT` | Add new records |
| **R**ead | `SELECT` | Retrieve/query records |
| **U**pdate | `UPDATE` | Modify existing records |
| **D**elete | `DELETE` | Remove records |

These operations form the foundation of all database interactions.

# 2. Setup - Database Connection Helper

In [ ]:
import mysql.connector
from mysql.connector import Error

# Database configuration
DB_CONFIG = {
    'host': 'localhost',
    'user': 'root',
    'password': 'your_password',
    'database': 'test_db'
}

def get_connection():
    """Create and return a database connection."""
    try:
        connection = mysql.connector.connect(**DB_CONFIG)
        return connection
    except Error as e:
        print(f"Error: {e}")
        return None

print("Connection helper ready!")

## 2.1 Create Sample Table

In [ ]:
# Create a sample employees table for our CRUD examples

connection = get_connection()
cursor = connection.cursor()

# Drop table if exists (for clean start)
cursor.execute("DROP TABLE IF EXISTS employees")

# Create table
create_table = """
CREATE TABLE employees (
    emp_id INT PRIMARY KEY AUTO_INCREMENT,
    name VARCHAR(100) NOT NULL,
    email VARCHAR(100) UNIQUE,
    department VARCHAR(50),
    salary DECIMAL(10, 2),
    hire_date DATE,
    is_active BOOLEAN DEFAULT TRUE
)
"""

cursor.execute(create_table)
print("Table 'employees' created successfully!")

cursor.close()
connection.close()

# 3. CREATE - INSERT Operations

Adding new records to the database.

## 3.1 Insert Single Record

In [ ]:
connection = get_connection()
cursor = connection.cursor()

# Method 1: Simple INSERT with values directly (NOT RECOMMENDED - SQL Injection risk!)
# cursor.execute("INSERT INTO employees (name, email) VALUES ('John', 'john@email.com')")

# Method 2: Parameterized query using %s placeholders (RECOMMENDED)
insert_query = """
INSERT INTO employees (name, email, department, salary, hire_date)
VALUES (%s, %s, %s, %s, %s)
"""

# Data to insert
employee_data = ("John Doe", "john@email.com", "Engineering", 75000.00, "2024-01-15")

cursor.execute(insert_query, employee_data)

# IMPORTANT: Commit the transaction to save changes
connection.commit()

print(f"Record inserted successfully!")
print(f"Inserted ID: {cursor.lastrowid}")  # Get the auto-generated ID

cursor.close()
connection.close()

## 3.2 Insert Multiple Records

In [ ]:
connection = get_connection()
cursor = connection.cursor()

insert_query = """
INSERT INTO employees (name, email, department, salary, hire_date)
VALUES (%s, %s, %s, %s, %s)
"""

# List of tuples with multiple employees
employees_data = [
    ("Jane Smith", "jane@email.com", "Marketing", 65000.00, "2024-02-01"),
    ("Bob Wilson", "bob@email.com", "Engineering", 80000.00, "2023-11-15"),
    ("Alice Brown", "alice@email.com", "HR", 55000.00, "2024-03-01"),
    ("Charlie Davis", "charlie@email.com", "Engineering", 72000.00, "2023-08-20"),
    ("Eva Martinez", "eva@email.com", "Marketing", 68000.00, "2024-01-10")
]

# executemany() for multiple inserts - more efficient than loop
cursor.executemany(insert_query, employees_data)

connection.commit()

print(f"{cursor.rowcount} records inserted successfully!")

cursor.close()
connection.close()

## 3.3 SQL Injection Prevention

**NEVER** concatenate user input directly into SQL queries!

In [ ]:
# ==========================================
# DANGEROUS - SQL INJECTION VULNERABILITY!
# ==========================================

user_input = "John'; DROP TABLE employees; --"  # Malicious input

# BAD - String concatenation (NEVER DO THIS!)
# query = f"SELECT * FROM employees WHERE name = '{user_input}'"
# This would execute: SELECT * FROM employees WHERE name = 'John'; DROP TABLE employees; --'


# ==========================================
# SAFE - Parameterized Queries
# ==========================================

# GOOD - Using placeholders (ALWAYS DO THIS!)
safe_query = "SELECT * FROM employees WHERE name = %s"
# The input is treated as data, not SQL code
# cursor.execute(safe_query, (user_input,))

print("Always use parameterized queries to prevent SQL injection!")
print("Use %s placeholders and pass data as a tuple.")

# 4. READ - SELECT Operations

Retrieving data from the database.

## 4.1 Fetch All Records

After executing a SELECT query, you need to fetch the results to actually get the data into your program.

**fetchall()** - Retrieves all rows from the query result as a list of tuples.

**fetchone()** - Retrieves only the first row as a single tuple.

**fetchmany(size)** - Retrieves a specific number of rows.

Note: cursor.execute() runs the query but doesn't return the data. You must use a fetch method to get the results.

In [ ]:
connection = get_connection()
cursor = connection.cursor()

# Select all columns
cursor.execute("SELECT * FROM employees")

# fetchall() returns list of tuples
results = cursor.fetchall()

print("All Employees:")
print("-" * 80)
for row in results:
    print(f"ID: {row[0]}, Name: {row[1]}, Email: {row[2]}, Dept: {row[3]}, Salary: ${row[4]}")

print(f"\nTotal records: {len(results)}")

cursor.close()
connection.close()

## 4.2 Fetch as Dictionary

In [ ]:
connection = get_connection()

# dictionary=True returns results as dictionaries instead of tuples
cursor = connection.cursor(dictionary=True)

cursor.execute("SELECT * FROM employees")
results = cursor.fetchall()

print("Employees as dictionaries:")
print("-" * 80)
for emp in results:
    # Access columns by name instead of index
    print(f"Name: {emp['name']}, Department: {emp['department']}, Salary: ${emp['salary']}")

cursor.close()
connection.close()

## 4.3 Select Specific Columns

In [ ]:
connection = get_connection()
cursor = connection.cursor(dictionary=True)

# Select only specific columns
cursor.execute("SELECT name, email, salary FROM employees")
results = cursor.fetchall()

print("Name, Email, and Salary only:")
for emp in results:
    print(emp)

cursor.close()
connection.close()

## 4.4 WHERE Clause - Filtering

In [ ]:
connection = get_connection()
cursor = connection.cursor(dictionary=True)

# ==========================================
# Filter by department
# ==========================================
department = "Engineering"
cursor.execute("SELECT * FROM employees WHERE department = %s", (department,))
print(f"\nEngineering Department:")
for emp in cursor.fetchall():
    print(f"  {emp['name']} - ${emp['salary']}")

# ==========================================
# Filter by salary range
# ==========================================
min_salary = 70000
cursor.execute("SELECT * FROM employees WHERE salary >= %s", (min_salary,))
print(f"\nSalary >= ${min_salary}:")
for emp in cursor.fetchall():
    print(f"  {emp['name']} - ${emp['salary']}")

# ==========================================
# Multiple conditions with AND/OR
# ==========================================
cursor.execute("""
    SELECT * FROM employees 
    WHERE department = %s AND salary > %s
""", ("Engineering", 70000))
print(f"\nEngineering with salary > 70000:")
for emp in cursor.fetchall():
    print(f"  {emp['name']} - ${emp['salary']}")

cursor.close()
connection.close()

## 4.5 Comparison Operators

In [ ]:
connection = get_connection()
cursor = connection.cursor(dictionary=True)

# ==========================================
# COMPARISON OPERATORS
# ==========================================

# Equals: =
cursor.execute("SELECT name FROM employees WHERE department = 'Marketing'")
print("Marketing:", [e['name'] for e in cursor.fetchall()])

# Not equals: != or <>
cursor.execute("SELECT name FROM employees WHERE department != 'Engineering'")
print("Not Engineering:", [e['name'] for e in cursor.fetchall()])

# Greater than: >
cursor.execute("SELECT name, salary FROM employees WHERE salary > 70000")
print("\nSalary > 70000:")
for e in cursor.fetchall():
    print(f"  {e['name']}: ${e['salary']}")

# Less than or equal: <=
cursor.execute("SELECT name, salary FROM employees WHERE salary <= 65000")
print("\nSalary <= 65000:")
for e in cursor.fetchall():
    print(f"  {e['name']}: ${e['salary']}")

# BETWEEN (inclusive)
cursor.execute("SELECT name, salary FROM employees WHERE salary BETWEEN 60000 AND 75000")
print("\nSalary between 60k-75k:")
for e in cursor.fetchall():
    print(f"  {e['name']}: ${e['salary']}")

# IN (matches any in list)
cursor.execute("SELECT name, department FROM employees WHERE department IN ('Engineering', 'HR')")
print("\nEngineering or HR:")
for e in cursor.fetchall():
    print(f"  {e['name']}: {e['department']}")

# LIKE (pattern matching)
# % = any characters, _ = single character
cursor.execute("SELECT name FROM employees WHERE name LIKE 'J%'")  # Starts with J
print("\nNames starting with J:", [e['name'] for e in cursor.fetchall()])

cursor.execute("SELECT name FROM employees WHERE email LIKE '%@email.com'")  # Ends with @email.com
print("Email ending with @email.com:", [e['name'] for e in cursor.fetchall()])

cursor.close()
connection.close()

## 4.6 ORDER BY and LIMIT

In [ ]:
connection = get_connection()
cursor = connection.cursor(dictionary=True)

# ==========================================
# ORDER BY - Sorting results
# ==========================================

# Ascending (default)
cursor.execute("SELECT name, salary FROM employees ORDER BY salary")
print("Salary Ascending:")
for e in cursor.fetchall():
    print(f"  {e['name']}: ${e['salary']}")

# Descending
cursor.execute("SELECT name, salary FROM employees ORDER BY salary DESC")
print("\nSalary Descending:")
for e in cursor.fetchall():
    print(f"  {e['name']}: ${e['salary']}")

# Multiple columns
cursor.execute("SELECT name, department, salary FROM employees ORDER BY department, salary DESC")
print("\nBy Department, then Salary (desc):")
for e in cursor.fetchall():
    print(f"  {e['department']}: {e['name']} - ${e['salary']}")

# ==========================================
# LIMIT - Restricting results
# ==========================================

# Get top 3 highest paid
cursor.execute("SELECT name, salary FROM employees ORDER BY salary DESC LIMIT 3")
print("\nTop 3 Highest Paid:")
for e in cursor.fetchall():
    print(f"  {e['name']}: ${e['salary']}")

# LIMIT with OFFSET for pagination
# LIMIT rows OFFSET skip
cursor.execute("SELECT name FROM employees ORDER BY name LIMIT 2 OFFSET 2")
print("\nPage 2 (skip 2, get 2):")
for e in cursor.fetchall():
    print(f"  {e['name']}")

cursor.close()
connection.close()

## 4.7 Aggregate Functions

In [ ]:
connection = get_connection()
cursor = connection.cursor(dictionary=True)

# ==========================================
# AGGREGATE FUNCTIONS
# ==========================================

# COUNT - Number of records
cursor.execute("SELECT COUNT(*) as total FROM employees")
print(f"Total employees: {cursor.fetchone()['total']}")

# COUNT with condition
cursor.execute("SELECT COUNT(*) as count FROM employees WHERE department = 'Engineering'")
print(f"Engineering count: {cursor.fetchone()['count']}")

# SUM - Total of a column
cursor.execute("SELECT SUM(salary) as total_salary FROM employees")
print(f"Total salary: ${cursor.fetchone()['total_salary']}")

# AVG - Average
cursor.execute("SELECT AVG(salary) as avg_salary FROM employees")
result = cursor.fetchone()['avg_salary']
print(f"Average salary: ${result:.2f}")

# MIN and MAX
cursor.execute("SELECT MIN(salary) as min_sal, MAX(salary) as max_sal FROM employees")
result = cursor.fetchone()
print(f"Salary range: ${result['min_sal']} - ${result['max_sal']}")

# ==========================================
# GROUP BY - Aggregate by category
# ==========================================

cursor.execute("""
    SELECT department, 
           COUNT(*) as emp_count, 
           AVG(salary) as avg_salary
    FROM employees 
    GROUP BY department
""")
print("\nBy Department:")
for row in cursor.fetchall():
    print(f"  {row['department']}: {row['emp_count']} employees, avg ${row['avg_salary']:.2f}")

# HAVING - Filter on aggregates (like WHERE but for GROUP BY)
cursor.execute("""
    SELECT department, AVG(salary) as avg_salary
    FROM employees 
    GROUP BY department
    HAVING AVG(salary) > 65000
""")
print("\nDepartments with avg salary > 65000:")
for row in cursor.fetchall():
    print(f"  {row['department']}: ${row['avg_salary']:.2f}")

cursor.close()
connection.close()

## 4.8 Fetch Methods Comparison

In [ ]:
connection = get_connection()
cursor = connection.cursor()

cursor.execute("SELECT * FROM employees")

# ==========================================
# fetchone() - Get single row
# ==========================================
first_row = cursor.fetchone()
print(f"First row: {first_row}")

# ==========================================
# fetchmany(n) - Get n rows
# ==========================================
next_two = cursor.fetchmany(2)
print(f"\nNext 2 rows: {next_two}")

# ==========================================
# fetchall() - Get remaining rows
# ==========================================
remaining = cursor.fetchall()
print(f"\nRemaining rows: {remaining}")

cursor.close()
connection.close()

print("\n" + "="*50)
print("TIP: For large datasets, use fetchone() or fetchmany()")
print("to avoid loading everything into memory at once.")

# 5. UPDATE Operations

Modifying existing records.

## 5.1 Update Single Record

In [ ]:
connection = get_connection()
cursor = connection.cursor(dictionary=True)

# Show current state
cursor.execute("SELECT emp_id, name, salary FROM employees WHERE name = 'John Doe'")
print(f"Before: {cursor.fetchone()}")

# Update salary for specific employee
update_query = """
UPDATE employees 
SET salary = %s 
WHERE name = %s
"""

new_salary = 85000.00
employee_name = "John Doe"

cursor.execute(update_query, (new_salary, employee_name))
connection.commit()  # Don't forget to commit!

print(f"\nRows affected: {cursor.rowcount}")

# Verify the update
cursor.execute("SELECT emp_id, name, salary FROM employees WHERE name = 'John Doe'")
print(f"After: {cursor.fetchone()}")

cursor.close()
connection.close()

## 5.2 Update Multiple Columns

In [ ]:
connection = get_connection()
cursor = connection.cursor(dictionary=True)

# Update multiple columns at once
update_query = """
UPDATE employees 
SET department = %s, salary = %s 
WHERE emp_id = %s
"""

# Update employee ID 2
cursor.execute(update_query, ("Sales", 70000.00, 2))
connection.commit()

# Verify
cursor.execute("SELECT * FROM employees WHERE emp_id = 2")
print(f"Updated employee: {cursor.fetchone()}")

cursor.close()
connection.close()

## 5.3 Update Multiple Records

In [ ]:
connection = get_connection()
cursor = connection.cursor(dictionary=True)

# Give 10% raise to all Engineering employees
update_query = """
UPDATE employees 
SET salary = salary * 1.10 
WHERE department = %s
"""

cursor.execute(update_query, ("Engineering",))
connection.commit()

print(f"Rows affected: {cursor.rowcount}")

# Verify
cursor.execute("SELECT name, salary FROM employees WHERE department = 'Engineering'")
print("\nEngineering salaries after 10% raise:")
for emp in cursor.fetchall():
    print(f"  {emp['name']}: ${emp['salary']:.2f}")

cursor.close()
connection.close()

## 5.4 Update with CASE Statement

In [ ]:
connection = get_connection()
cursor = connection.cursor(dictionary=True)

# Different raises based on department
update_query = """
UPDATE employees 
SET salary = CASE 
    WHEN department = 'Engineering' THEN salary * 1.05
    WHEN department = 'Marketing' THEN salary * 1.03
    ELSE salary * 1.02
END
"""

cursor.execute(update_query)
connection.commit()

print(f"Rows affected: {cursor.rowcount}")

# Show results
cursor.execute("SELECT name, department, salary FROM employees ORDER BY department")
print("\nUpdated salaries:")
for emp in cursor.fetchall():
    print(f"  {emp['department']}: {emp['name']} - ${emp['salary']:.2f}")

cursor.close()
connection.close()

# 6. DELETE Operations

Removing records from the database.

## 6.1 Delete Single Record

In [ ]:
connection = get_connection()
cursor = connection.cursor(dictionary=True)

# First, let's add a record to delete
cursor.execute("""
    INSERT INTO employees (name, email, department, salary) 
    VALUES ('Temp Worker', 'temp@email.com', 'Temp', 30000)
""")
connection.commit()
temp_id = cursor.lastrowid
print(f"Added temp employee with ID: {temp_id}")

# Delete by ID
delete_query = "DELETE FROM employees WHERE emp_id = %s"
cursor.execute(delete_query, (temp_id,))
connection.commit()

print(f"Deleted {cursor.rowcount} record(s)")

# Verify deletion
cursor.execute("SELECT * FROM employees WHERE emp_id = %s", (temp_id,))
result = cursor.fetchone()
print(f"Verification (should be None): {result}")

cursor.close()
connection.close()

## 6.2 Delete with Conditions

In [ ]:
connection = get_connection()
cursor = connection.cursor()

# Add some records to delete
cursor.executemany("""
    INSERT INTO employees (name, email, department, salary, is_active) 
    VALUES (%s, %s, %s, %s, %s)
""", [
    ('Inactive1', 'inactive1@email.com', 'Old', 40000, False),
    ('Inactive2', 'inactive2@email.com', 'Old', 35000, False),
])
connection.commit()
print(f"Added {cursor.rowcount} inactive employees")

# Delete all inactive employees
delete_query = "DELETE FROM employees WHERE is_active = %s"
cursor.execute(delete_query, (False,))
connection.commit()

print(f"Deleted {cursor.rowcount} inactive employees")

cursor.close()
connection.close()

## 6.3 Safe Delete Pattern

In [ ]:
connection = get_connection()
cursor = connection.cursor(dictionary=True)

def safe_delete(cursor, connection, table, condition, params):
    """
    Safely delete records with preview.
    Shows what will be deleted before actually deleting.
    """
    # First, preview what will be deleted
    preview_query = f"SELECT * FROM {table} WHERE {condition}"
    cursor.execute(preview_query, params)
    records = cursor.fetchall()
    
    if not records:
        print("No records match the condition.")
        return 0
    
    print(f"Records to be deleted ({len(records)}):")
    for record in records:
        print(f"  {record}")
    
    # In production, you might want user confirmation here
    # confirm = input("Delete these records? (yes/no): ")
    # if confirm.lower() != 'yes':
    #     return 0
    
    # Perform the delete
    delete_query = f"DELETE FROM {table} WHERE {condition}"
    cursor.execute(delete_query, params)
    connection.commit()
    
    return cursor.rowcount

# Example usage (won't actually delete in this demo)
# deleted = safe_delete(cursor, connection, 'employees', 'department = %s', ('Temp',))

print("Safe delete function defined!")

cursor.close()
connection.close()

## 6.4 TRUNCATE vs DELETE

In [ ]:
# ==========================================
# DELETE vs TRUNCATE
# ==========================================

# DELETE - removes records one by one
# - Can use WHERE clause
# - Triggers are fired
# - Can be rolled back
# - Slower for large tables
# - Auto-increment continues from last value

# DELETE FROM employees WHERE department = 'Temp';  -- Delete some
# DELETE FROM employees;  -- Delete all (but one by one)

# TRUNCATE - removes all records at once
# - No WHERE clause (removes ALL)
# - Triggers NOT fired
# - Cannot be rolled back
# - Very fast
# - Resets auto-increment to 1

# TRUNCATE TABLE employees;  -- Remove ALL records, reset ID

print("DELETE vs TRUNCATE:")
print("="*50)
print("DELETE:")
print("  - Use for selective deletion")
print("  - Slower but safer")
print("  - Can rollback")
print("\nTRUNCATE:")
print("  - Use to clear entire table")
print("  - Very fast")
print("  - Cannot rollback - USE WITH CAUTION!")

# 7. Transactions

Transactions ensure multiple operations complete together or not at all (all-or-nothing).

In [ ]:
connection = get_connection()
cursor = connection.cursor()

try:
    # Start transaction (autocommit is off by default)
    
    # Operation 1: Deduct from one account
    cursor.execute("""
        UPDATE employees SET salary = salary - 5000 
        WHERE emp_id = 1
    """)
    
    # Operation 2: Add to another account
    cursor.execute("""
        UPDATE employees SET salary = salary + 5000 
        WHERE emp_id = 2
    """)
    
    # If both succeed, commit
    connection.commit()
    print("Transaction committed successfully!")
    
except Exception as e:
    # If any error, rollback all changes
    connection.rollback()
    print(f"Transaction rolled back due to: {e}")

finally:
    cursor.close()
    connection.close()

# 8. Complete CRUD Example

In [ ]:
import mysql.connector
from mysql.connector import Error

class EmployeeCRUD:
    """Complete CRUD operations for employees table."""
    
    def __init__(self, config):
        self.config = config
    
    def _get_connection(self):
        return mysql.connector.connect(**self.config)
    
    # CREATE
    def create(self, name, email, department, salary):
        """Add a new employee."""
        query = """
            INSERT INTO employees (name, email, department, salary)
            VALUES (%s, %s, %s, %s)
        """
        with self._get_connection() as conn:
            with conn.cursor() as cursor:
                cursor.execute(query, (name, email, department, salary))
                conn.commit()
                return cursor.lastrowid
    
    # READ
    def get_all(self):
        """Get all employees."""
        with self._get_connection() as conn:
            with conn.cursor(dictionary=True) as cursor:
                cursor.execute("SELECT * FROM employees")
                return cursor.fetchall()
    
    def get_by_id(self, emp_id):
        """Get employee by ID."""
        with self._get_connection() as conn:
            with conn.cursor(dictionary=True) as cursor:
                cursor.execute("SELECT * FROM employees WHERE emp_id = %s", (emp_id,))
                return cursor.fetchone()
    
    def search(self, **kwargs):
        """Search employees by any field."""
        conditions = [f"{k} = %s" for k in kwargs.keys()]
        query = f"SELECT * FROM employees WHERE {' AND '.join(conditions)}"
        with self._get_connection() as conn:
            with conn.cursor(dictionary=True) as cursor:
                cursor.execute(query, tuple(kwargs.values()))
                return cursor.fetchall()
    
    # UPDATE
    def update(self, emp_id, **kwargs):
        """Update employee fields."""
        if not kwargs:
            return 0
        set_clause = ", ".join([f"{k} = %s" for k in kwargs.keys()])
        query = f"UPDATE employees SET {set_clause} WHERE emp_id = %s"
        values = tuple(kwargs.values()) + (emp_id,)
        with self._get_connection() as conn:
            with conn.cursor() as cursor:
                cursor.execute(query, values)
                conn.commit()
                return cursor.rowcount
    
    # DELETE
    def delete(self, emp_id):
        """Delete employee by ID."""
        with self._get_connection() as conn:
            with conn.cursor() as cursor:
                cursor.execute("DELETE FROM employees WHERE emp_id = %s", (emp_id,))
                conn.commit()
                return cursor.rowcount


# Example usage:
# crud = EmployeeCRUD(DB_CONFIG)
# 
# # Create
# new_id = crud.create("New Employee", "new@email.com", "IT", 60000)
# 
# # Read
# all_employees = crud.get_all()
# one_employee = crud.get_by_id(1)
# engineering = crud.search(department="Engineering")
# 
# # Update
# crud.update(new_id, salary=65000, department="DevOps")
# 
# # Delete
# crud.delete(new_id)

print("EmployeeCRUD class defined - ready to use!")

# 9. Summary

## CRUD Operations Quick Reference

| Operation | SQL | Python Method |
|-----------|-----|---------------|
| **Create** | `INSERT INTO table (cols) VALUES (vals)` | `cursor.execute()` + `commit()` |
| **Read** | `SELECT cols FROM table WHERE condition` | `cursor.execute()` + `fetchall/one/many()` |
| **Update** | `UPDATE table SET col=val WHERE condition` | `cursor.execute()` + `commit()` |
| **Delete** | `DELETE FROM table WHERE condition` | `cursor.execute()` + `commit()` |

## Best Practices

1. **Always use parameterized queries** - Prevent SQL injection
2. **Always commit after changes** - INSERT, UPDATE, DELETE need commit
3. **Use transactions** - For multiple related operations
4. **Close connections** - Use context managers (`with`)
5. **Handle errors** - Use try-except blocks
6. **Validate before delete** - Preview records to be deleted